# Clickbait Title/Body Logistic Regression Training
`title/body` split CSV를 사용해 Logistic Regression을 학습합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/sw_project/ai_features/clickbait_detection'
LOGREG_SRC = os.path.join(DRIVE_ROOT, 'clickbait_logreg_tfidf')
LOGREG_WORK = '/content/clickbait_logreg_tfidf'
DATA_SRC = os.path.join(DRIVE_ROOT, 'data')

assert os.path.exists(LOGREG_SRC), f'Not found: {LOGREG_SRC}'
assert os.path.exists(DATA_SRC), f'Not found: {DATA_SRC}'

if os.path.exists(LOGREG_WORK):
    shutil.rmtree(LOGREG_WORK)
shutil.copytree(LOGREG_SRC, LOGREG_WORK)
if os.path.exists(os.path.join(LOGREG_WORK, 'data')):
    shutil.rmtree(os.path.join(LOGREG_WORK, 'data'))
shutil.copytree(DATA_SRC, os.path.join(LOGREG_WORK, 'data'), dirs_exist_ok=True)

os.chdir(LOGREG_WORK)
!pip -q install -r requirements.txt


In [ ]:
!python3 -u train.py \
  --train-data data/train.csv \
  --valid-data data/valid.csv \
  --test-data data/test.csv \
  --title-col title \
  --body-col body \
  --label-col label \
  --c 10.0 \
  --model-out models/logreg_tfidf_title_body.joblib


In [ ]:
!ls -lah models | sed -n '1,120p'


In [ ]:
import json
from pathlib import Path

import joblib
from sklearn.metrics import accuracy_score, f1_score

from src.data_utils import load_training_data

model_path = "models/logreg_tfidf_title_body.joblib"
metrics_path = Path(model_path).with_name("metrics.json")

model = joblib.load(model_path)

results = {}
for split_name, csv_path in [
    ("valid", "data/valid.csv"),
    ("test", "data/test.csv"),
]:
    x, y = load_training_data(csv_path, title_col="title", body_col="body", label_col="label")
    pred = model.predict(x)
    results[split_name] = {
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro")),
        "weighted_f1": float(f1_score(y, pred, average="weighted")),
    }

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"saved -> {metrics_path}")
print(json.dumps(results, ensure_ascii=False, indent=2))
